# Standard simulator workbench

This is the continuing workbench for the complete Standard simulator target. **The full game remains unfinished.** Written effects and passing scenario checks are separate measures.

Prior versions are backed up; outputs here are cleared. This notebook does not train or run match batches. **Restart the kernel before running changed engine code.** All execution is yours to start.

Version 0.14 combines selector and lifecycle work with explicit generated-pool validation: **291 written implementations; 241 prepared checks**. Restart the kernel before running the checks.

Reload from disk (choose **Revert** if prompted), restart the kernel and run all cells. Paste only the final summary and any failure tracebacks. No tests were run by the assistant.


In [ ]:
from pathlib import Path
import sys, json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "expanded" / "game.py").exists()), None)
if ROOT is None: raise RuntimeError("Open this notebook inside death-knight-lab.")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from expanded import Game
from expanded.status import readiness, run_rule_fixtures
if Game.VERSION != "all-class-experimental-0.14":
    raise RuntimeError("Restart the kernel to load the updated engine.")
state = readiness()
print("Engine:", state["engine"])
print("Effects written:", state["effects_written"], "/", state["catalog_cards"])
print("Still missing:", len(state["missing_cards"]))
print("Current-code checks passed:", state["fixtures_passed_for_current_code"])
print("Full Standard ready:", state["full_standard_ready"])


## Run the prepared rules checks

This discovers all prepared rules checks, including damage ordering, pause/resume, and shared card-play scenarios. The progress bar shows the actual count. These checks run only when you start this cell; they do not train or run match batches. Passing does not certify full Standard behavior.


In [ ]:
from IPython.display import display, HTML
progress_display = display(HTML("<progress max='1' value='0'></progress> Preparing checks…"), display_id=True)
def show_progress(done, total, failures, errors):
    progress_display.update(HTML(
        f"<progress max='{max(1,total)}' value='{done}' style='width:360px'></progress> "
        f"{done}/{total} checks · {failures} failures · {errors} errors"
    ))
validation = run_rule_fixtures(progress=show_progress)
print("Passed:", validation["success"])
print("Checks run:", validation["tests_run"])
if not validation["success"]:
    raise RuntimeError("Checks failed. Keep this output for diagnosis; no training has started.")


## Save the current completion report

Every catalog card appears in the audit, including those still missing. The dependency hints organize development; they are not implementations.


In [ ]:
state = readiness()
folder = ROOT / "runs" / "expanded_validation"
folder.mkdir(parents=True, exist_ok=True)
path = folder / "readiness.json"
temporary = path.with_suffix(".tmp")
temporary.write_text(json.dumps(state, indent=2) + "\n")
temporary.replace(path)
audit = json.loads((ROOT / "expanded" / "catalog_audit.json").read_text())
print("Saved:", path)
print("Catalog entries tracked:", len(audit["cards"]))
print("Remaining effects:", len(state["missing_cards"]))
print("Full Standard ready:", state["full_standard_ready"])
for task in state["remaining"]: print("•", task)


## Completion boundary

Serious full-Standard training stays blocked until the missing effects, interaction systems, generation pools, legality rules, and independent validation are complete. The saved original model is still usable in its original subset, but it cannot rank full-Standard decks.

See docs/standard-completion.md for the exact remaining work.
